## Importar librerías

In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import os
import time
import multiprocessing as mp
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, recall_score

## Descargar el conjunto de datos

In [ ]:
# Descargar el conjunto de datos completo
dataset_ruta_kaggle = kagglehub.dataset_download(
    "meowmeowmeowmeowmeow/gtsrb-german-traffic-sign"
)

print("\nArchivos y directorios en el conjunto de datos:")
for ruta_raiz, dircs, archivos in os.walk(dataset_ruta_kaggle):
    for nombre in dircs:
        print(os.path.join(ruta_raiz, nombre) + '/')
    for nombre in archivos:
        print(os.path.join(ruta_raiz, nombre))

In [ ]:
# Listamos los directorios que se descargaron
print(os.listdir('/kaggle/input/gtsrb-german-traffic-sign/'))

## Convertir en arreglos

In [ ]:
datos = []
etiquetas = []
clases = 43

# Carga de imágenes y etiquetas
for i in range(clases):
    ruta = os.ruta.join('/kaggle/input/gtsrb-german-traffic-sign/', 'train', str(i))
    images = os.listdir(ruta)

    for a in images:
        try:
            img = Image.open(ruta + '/' + a)
            img = img.resize((30, 30))
            img = np.array(img)
            datos.append(img)
            etiquetas.append(i)
        except:
            print("Error al cargar img")

datos = np.array(datos)
etiquetas = np.array(etiquetas)

## Separar el conjunto de datos

In [ ]:
print(datos.shape, etiquetas.shape)

X_train, X_test, y_train, y_test = train_test_split(
    datos, etiquetas, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

# Aplanar las imágenes para SVM
X_train= X_train.reshape(X_train.shape[0], -1)
X_test= X_test.reshape(X_test.shape[0], -1)

# normalizar los píxeles a [0, 1]
X_train= X_train/ 255.0
X_test= X_test / 255.0

print("Dimensiones después de aplanar:")
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

## Entrenar el modelo

### SVM


In [ ]:
# Parámetros fijos del modelo
C_value= 0.1
kernel_value='rbf'
gamma_value= 0.1

def evaluar_svm():
    svm_clf = SVC(C=C_value, kernel=kernel_value, gamma=gamma_value)
    svm_clf.fit(X_train, y_train)
    pred = svm_clf.predict(X_test)
    acc  = accuracy_score(y_test, pred)
    return acc, pred


# --parte secuencial
start_seq = time.time()
acc_seq, pred_seq = evaluar_svm()
t_seq = time.time() - start_seq

# --parte paralelo
N_paralelo = 4  #número de veces que se ejecuta la misma tarea en paralelo

start_par = time.time()
with mp.Pool(processes=4) as pool:
    resultados_par = pool.map(lambda _: evaluar_svm(), range(N_paralelo))
t_par = time.time() - start_par

acc_par, pred_par = resultados_par[0]

# -- Métricas de paralelización 
speedup    = t_seq / t_par if t_par > 0 else float('inf')
eficiencia = (speedup / 4) * 100   # E = S(p) / p

In [ ]:
# --Resultados finales
print("-- Métricas de paralelización\n")
print(f"Tiempo secuencial: {t_seq:.2f} seg")
print(f"Tiempo paralelo: {t_par:.2f} seg")
print(f"Speedup: {speedup:.2f}")
print(f"Eficiencia: {eficiencia:.2f}%")
print()
print("-- Métricas del modelo (resultado secuencial)\n")
print(f"Exactitud: {acc_seq:.4f}")
print(f"F1 (macro): {f1_score(y_test, pred_seq, average='macro'):.4f}")
print(f"Recall: {recall_score(y_test, pred_seq, average='macro'):.4f}")